## 1. 데이터 준비 및 전체 구조 파악

In [1]:
# 라이브러리 불러오기
import pandas as pd
import numpy as np

In [3]:
# 데이터 불러오기
df = pd.read_csv('../dataset/raw/hotel_bookings.csv')
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [4]:
# 데이터 정보
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

## 2. 결측치 처리

In [5]:
# 결측치 확인
df.isna().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

### - 변수 종류에 따른 결측치 처리
- children: 자녀수 -----> 0
- country: 국적 -----> 삭제
- agent: 예약을 진행한 여행사의 ID -----> 0: 새 값 추가, 직접 예약
- company: 예약을 진행했거나 예약 비용을 지불한 회사/기관의 ID -----> 0: 새 값 추가, 직접 예약

In [ ]:
# 결측치 처리
missing_cols_drop = ['country']
missing_cols_fill = ['children', 'agent', 'company']

df.dropna(subset=missing_cols_drop[0], axis=0, inplace=True)
df[missing_cols_fill] = df[missing_cols_fill].fillna(0)

df.isna().sum()

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_month         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
agent                             0
company                           0
days_in_waiting_list              0
customer_type                     0
adr                         

## 3. 도착일 관련 데이터

In [ ]:
# 대상: 

# 3   arrival_date_year               119390 non-null  int64

# 도착일 - 년

# 4   arrival_date_month              119390 non-null  str

# 도착일 - 월

# 5   arrival_date_week_number        119390 non-null  int64

# 도착일 - 주차

# 6   arrival_date_day_of_month       119390 non-null  int64

# 도착일 - 일

# 삭제: arrival_date_week_number
del_arrival_date_cols = ['arrival_date_year', 'arrival_date_day_of_month']

df.drop(del_arrival_date_cols, axis=1, inplace=True)

## 4. 투숙객 현황 관련 데이터

9   adults                          119390 non-null  int64

성인 수

10  children                        119386 non-null  float64

자녀 수

11  babies                          119390 non-null  int64

아기 수

26  customer_type                   119390 non-null  str

예약 유형(다음 네 가지 범주 중 하나라고 가정):

- 계약 - 예약에 할당량 또는 기타 유형의 계약이 포함된 경우;
- 그룹 예약 – 예약이 그룹과 연결된 경우;
- 개별 예약 – 단체 예약이나 계약에 포함되지 않고, 다른 개별 예약과 연계되지 않은 예약.
- 일시적 예약 - 예약이 일시적이지만, 적어도 다른 일시적 예약과 연관된 경우

In [11]:
cs_type_cols = ['adults', 'children', 'babies', 'customer_type']
df[cs_type_cols].describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
adults,118902.0,NaN,NaN,NaN,1.858404,0.578576,0.0,2.0,2.0,2.0,55.0
children,118902.0,NaN,NaN,NaN,0.104203,0.399166,0.0,0.0,0.0,0.0,10.0
babies,118902.0,NaN,NaN,NaN,0.007948,0.097379,0.0,0.0,0.0,0.0,10.0
customer_type,118902,4,Transient,89174,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
df.loc[df['adults'] == 0][cs_type_cols]

,adults,children,babies,customer_type
2224,0,0.0,0,Transient-Party
2409,0,0.0,0,Transient
3181,0,0.0,0,Transient-Party
3684,0,0.0,0,Transient-Party
3708,0,0.0,0,Transient-Party
...,...,...,...,...
117204,0,2.0,0,Transient
117274,0,2.0,0,Transient
117303,0,2.0,0,Transient
117453,0,2.0,0,Transient


In [ ]:
# - 계약 - 예약에 할당량 또는 기타 유형의 계약이 포함된 경우;Contract            
# - 그룹 예약 – 예약이 그룹과 연결된 경우;Group                
# - 개별 예약 – 단체 예약이나 계약에 포함되지 않고, 다른 개별 예약과 연계되지 않은 예약.
# - 일시적 예약 - 예약이 일시적이지만, 적어도 다른 일시적 예약과 연관된 경우
df['customer_type'].value_counts()

customer_type
Transient          89174
Transient-Party    25082
Contract            4076
Group                570
Name: count, dtype: int64

customer_type에 유형 구성이

In [19]:
df[cs_type_cols].info()

<class 'pandas.DataFrame'>
Index: 118902 entries, 0 to 119389
Data columns (total 4 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   adults         118902 non-null  int64  
 1   children       118902 non-null  float64
 2   babies         118902 non-null  int64  
 3   customer_type  118902 non-null  str    
dtypes: float64(1), int64(2), str(1)
memory usage: 4.5 MB


## 5. 객실 유형 관련 데이터

In [10]:
# diff_reserved_room_type 추가: 예약한 방과 배정받은 방이 다른 경우
# reserved_room_type, assigned_room_type 열 조합
# 각 열은 객실 유형을 코드로 나타낸 값으로 실제 객실 유형은 알 수 없으므로 삭제함
room_type_cols = ['reserved_room_type', 'assigned_room_type']

df['diff_reserved_room_type'] = df[room_type_cols[0]] != df[room_type_cols[1]]

df.drop(room_type_cols, axis=1, inplace=True)

## 예약 유형 관련 데이터
14  market_segment                  119390 non-null  str

시장 부문 명칭. 분류 체계에서 "TA"는 "여행사(Travel Agents)"를, "TO"는 "여행사(Tour Operators)"를 의미합니다.

15  distribution_channel            119390 non-null  str

예약 유통 채널. "TA"는 "여행사"를, "TO"는 "여행 운영업체"를 의미합니다.

23  agent                           103050 non-null  float64

예약을 진행한 여행사의 ID

24  company                         6797 non-null    float64

예약을 진행했거나 예약 비용을 지불한 회사/기관의 ID입니다. 익명성 유지를 위해 직책 대신 ID를 표시합니다.

In [ ]:
cols = ['market_segment', 'distribution_channel', 'agent', 'company']


## 예약 상태 관련 데이터
30  reservation_status              119390 non-null  str

| 예약 최종 상태는 다음 세 가지 범주 중 하나로 가정합니다. |
| --- |
| 취소됨 – 고객이 예약을 취소했습니다. |
| 체크아웃 - 고객이 체크인은 했지만 이미 출발했습니다. |
| 노쇼 – 고객이 체크인하지 않았으며, 호텔에 그 이유를 알렸습니다. |

31  reservation_status_date         119390 non-null  str

마지막 상태가 설정된 날짜입니다. 이 변수는 *ReservationStatus* 변수와 함께 사용하여 예약이 취소된 시점 또는 고객이 호텔에서 체크아웃한 시점을 파악하는 데 사용할 수 있습니다.

## 마무리

In [ ]:
# 파일 저장
df.to_csv('../data/preprocessed/hotel_bookings.csv', index=False)